# CosmicML-Biodetect: Introduction to Exoplanet Biosignatures

**Learning Objectives:**
- Understand what exoplanets are and why they matter
- Learn about biosignatures and how we detect them
- Understand transmission spectroscopy physics
- Explore the CosmicML-Biodetect framework

**References:**
- Seager, S., Turner, E. L., Schafer, E., & Ford, E. B. (2005). Vegetation's Red Edge: A Possible Spectroscopic Biosignature of Extraterrestrial Plants. Astrobiology, 5(2), 372-390.
- Madhusudhan, N., Amin, M. A., & Kennedy, G. M. (2014). Architecture and Fate of Planetary Systems. Monthly Notices of the Royal Astronomical Society, 445(2), 1561-1598.

## Part 1: What Are Exoplanets?

Exoplanets are planets that orbit stars other than our Sun.

**Key Facts:**
- First exoplanet discovered: 1995 (51 Pegasi b)
- Current count: 5,500+ confirmed exoplanets
- ~10,000 more candidate exoplanets awaiting confirmation

**Why Care About Exoplanets?**
1. **Scientific curiosity:** Are we alone in the universe?
2. **Astrobiology:** Understanding conditions for life
3. **Future exploration:** Target selection for missions
4. **Fundamental physics:** Testing gravitational theories

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Exoplanet discovery statistics
years = np.array([1995, 2000, 2005, 2010, 2015, 2020, 2024])
discoveries = np.array([1, 50, 150, 450, 2000, 4400, 5500])

plt.figure(figsize=(12, 6))
plt.plot(years, discoveries, 'o-', linewidth=2, markersize=8, color='#2E86AB')
plt.fill_between(years, discoveries, alpha=0.3, color='#2E86AB')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Confirmed Exoplanets', fontsize=12)
plt.title('Exoplanet Discovery Timeline (1995-2024)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Exoplanet discoveries have grown exponentially!")
print(f"Growth rate: ~{(discoveries[-1] - discoveries[0])/(years[-1] - years[0]):.0f} per year")

## Part 2: What Are Biosignatures?

A **biosignature** is a chemical or physical characteristic that indicates the presence of biological activity.

### Earth's Biosignatures

On Earth, several chemicals indicate life:

| Species | Source | Why It Matters |
|---------|--------|----------------|
| O₂ (Oxygen) | Photosynthetic organisms | 21% of atmosphere, produced entirely by life |
| CH₄ (Methane) | Biological processes | Oxidizes in 10 years; sustained levels need life |
| O₃ (Ozone) | O₂ photochemistry | Indicator of oxygen-rich atmosphere |
| N₂O (Nitrous Oxide) | Microbial processes | Produced by bacteria in soil/ocean |
| Complex organics | Life processes | Molecules that require biological synthesis |

### The Challenge

Abiotic (non-biological) processes can also produce these molecules:
- O₂ can form from UV photolysis of CO₂ and H₂O
- CH₄ can come from geological processes
- O₃ forms naturally from atmospheric O₂

**Our goal:** Detect true biosignatures and distinguish from abiotic false positives

In [ ]:
# Compare biosignature abundances: Earth vs abiotic scenarios
import pandas as pd

scenarios = {
    'Species': ['O₂', 'CH₄', 'O₃', 'H₂O', 'CO₂', 'N₂'],
    'Earth': [0.21, 1.8e-6, 5e-6, 0.01, 0.0004, 0.78],
    'Venus-like': [1e-12, 1e-10, 1e-9, 0.03, 0.96, 0.03],
    'Early Earth': [1e-15, 1e-8, 1e-11, 0.02, 0.03, 0.8],
}

df = pd.DataFrame(scenarios)

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(df['Species']))
width = 0.25

ax.bar(x - width, df['Earth'], width, label='Modern Earth', alpha=0.8)
ax.bar(x, df['Venus-like'], width, label='Venus-like', alpha=0.8)
ax.bar(x + width, df['Early Earth'], width, label='Early Earth', alpha=0.8)

ax.set_ylabel('Mixing Ratio (fraction)', fontsize=12)
ax.set_title('Atmospheric Composition Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df['Species'])
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\nKey Insights:")
print(f"- Earth O₂: {df['Earth'][0]:.1%} (produced by photosynthesis)")
print(f"- Venus O₂: {df['Venus-like'][0]:.0e} (no life, abiotic processes)")
print(f"- Earth CH₄: {df['Earth'][1]:.1e} (maintained by biological sources)")
print(f"- Venus CH₄: {df['Venus-like'][1]:.1e} (no detectable methane)")

## Part 3: How Do We Observe Exoplanet Atmospheres?

### Transmission Spectroscopy

We cannot directly image exoplanet atmospheres. Instead, we use **transmission spectroscopy**:

**The Physics:**
1. Star emits light (broad spectrum)
2. Light passes through exoplanet's upper atmosphere
3. Atmosphere absorbs specific wavelengths (molecular fingerprints)
4. We observe dips in stellar brightness at these wavelengths

**Transit Depth Equation:**
$$d(\lambda) = \frac{(R_p + H_{eff}(\lambda))^2 - R_p^2}{R_*^2}$$

where:
- $R_p$ = planet radius
- $R_*$ = star radius  
- $H_{eff}$ = effective scale height (wavelength-dependent due to absorption)
- $d(\lambda)$ = observed transit depth (~0.01-0.1%)

**Why It's Hard:**
- Tiny signals: 0.01-0.1% changes in brightness
- Need extreme sensitivity (JWST, Keck)
- Multiple species overlap in absorption
- Clouds and haze mask true composition

In [ ]:
# Visualize transmission spectroscopy
import sys
sys.path.insert(0, '../src')

from cosmicml.atmosphere.simulator import AtmosphereSimulator, PlanetaryConfig

# Create Earth-analog configuration
earth_analog = PlanetaryConfig(
    planet_radius=1.0,  # Earth radii
    planet_mass=1.0,    # Earth masses
    star_temp=5778,     # Sun-like
    orbital_period=365,
    equilibrium_temp=288,  # ~15°C
    surface_gravity=9.81,  # m/s²
    stellar_radius=1.0,
)

# Earth-like composition
earth_composition = {
    "N2": 0.78,
    "O2": 0.21,
    "H2O": 0.01,
    "CO2": 0.0004,
}

# Generate spectrum
simulator = AtmosphereSimulator(earth_analog)
atmosphere = simulator.simulate_atmosphere(earth_composition)
wavelengths, transit_depths, uncertainties = simulator.generate_spectrum(
    atmosphere,
    noise_level=1e-4,
    add_noise=True,
)

# Plot spectrum
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(wavelengths, transit_depths * 100, 'b-', linewidth=2, label='Transit Depth')
ax.fill_between(
    wavelengths,
    (transit_depths - uncertainties) * 100,
    (transit_depths + uncertainties) * 100,
    alpha=0.3,
    label='Uncertainty (1σ)'
)

# Mark features
ax.axvline(1.1, color='red', linestyle='--', alpha=0.5, label='H₂O absorption')
ax.axvline(2.7, color='green', linestyle='--', alpha=0.5, label='CO₂ absorption')

ax.set_xlabel('Wavelength (μm)', fontsize=12)
ax.set_ylabel('Transit Depth (%)', fontsize=12)
ax.set_title('Earth-like Exoplanet Transmission Spectrum', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nSpectrum Properties:")
print(f"- Mean transit depth: {np.mean(transit_depths)*100:.3f}%")
print(f"- Peak transit depth: {np.max(transit_depths)*100:.3f}%")
print(f"- Wavelength range: {wavelengths[0]:.2f}-{wavelengths[-1]:.2f} μm")
print(f"- Measurement uncertainty: {np.mean(uncertainties)*100:.4f}% (JWST-like)")

## Part 4: Physics-Informed Neural Networks (PINNs)

### The Problem with Standard ML

Regular neural networks can:
- Learn patterns from data ✓
- Make fast predictions ✓
- But violate physical laws ✗

Example failures:
- Predict negative abundances (impossible)
- Violate conservation laws
- Extrapolate unphysically

### The Solution: Physics-Informed Neural Networks

We train networks to satisfy **both** data and physics:

$$L_{total} = L_{data} + \lambda \cdot L_{physics}$$

where:
- $L_{data} = \text{MSE}(\text{predicted}, \text{target})$ - fit observations
- $L_{physics} = L_{abundance} + L_{conservation} + L_{thermo}$ - enforce laws

**Physics Constraints:**
1. **Abundance:** $0 \leq x_i \leq 1$, $\sum x_i = 1$
2. **Conservation:** Element ratios stable
3. **Thermodynamic:** Gibbs relations satisfied

### Benefits
- Better generalization (physics acts as regularization)
- Interpretable predictions (respect known science)
- Less training data needed
- Uncertainty quantification

In [ ]:
# Demonstrate PINN benefits
import torch
from cosmicml.models import PINN

# Create PINN model
pinn = PINN(
    input_dim=512,
    latent_dim=64,
    output_dim=10,
    physics_weight=1.0,
)

# Create sample spectra
sample_spectra = torch.randn(4, 512)
target_compositions = torch.softmax(torch.randn(4, 10), dim=1)

# Forward pass
predictions = pinn(sample_spectra)

# Compute losses
total_loss, data_loss, physics_loss = pinn.compute_loss(
    sample_spectra,
    target_compositions,
)

print("\n=== PINN Loss Components ===")
print(f"Data Loss (MSE):      {data_loss.item():.4f}")
print(f"Physics Loss:         {physics_loss.item():.4f}")
print(f"Total Loss:           {total_loss.item():.4f}")

print("\n=== Prediction Properties ===")
print(f"Mean abundance:       {torch.mean(predictions).item():.4f}")
print(f"Sum (per sample):     {torch.sum(predictions, dim=1).mean().item():.4f} (should be 1)")
print(f"Min value:            {torch.min(predictions).item():.4f} (should be ≥ 0)")
print(f"Max value:            {torch.max(predictions).item():.4f} (should be ≤ 1)")

print("\n✓ Physics constraints automatically enforced!")

## Part 5: The CosmicML-Biodetect Framework

### Architecture Overview

```
Input Spectrum (512 wavelengths)
    ↓
[Atmosphere Simulator] → Generate synthetic data
    ↓
[PINN Model] → Learn from data + physics
    ├─ Encoder (512→64)
    ├─ Physics constraints (enforced)
    └─ Decoder (64→10 species)
    ↓
[Bayesian Inference] → Compute probabilities
    ↓
Output: Atmospheric composition + uncertainties
```

### Key Modules

1. **Atmosphere Simulator** - Physics engine
   - Radiative transfer
   - Chemical reactions
   - Spectrum generation

2. **PINN Model** - Neural network with constraints
   - Encoder/decoder architecture
   - Physics loss terms
   - Uncertainty quantification

3. **Data Pipeline** - Preprocessing and loading
   - Synthetic data generation
   - JWST data integration
   - Normalization

4. **Training** - Full ML pipeline
   - PyTorch Lightning
   - TensorBoard logging
   - Early stopping

5. **Inference** - Biosignature detection
   - Posterior estimation
   - Credible intervals
   - Life probability scores

In [ ]:
# End-to-end example
print("\n" + "="*60)
print("CosmicML-Biodetect: End-to-End Workflow")
print("="*60)

print("\n1. GENERATE SYNTHETIC ATMOSPHERE")
print("-" * 60)

# Create diverse planet
earth_with_oxygen = PlanetaryConfig(
    planet_radius=1.2,
    planet_mass=1.5,
    star_temp=5000,
    orbital_period=350,
    equilibrium_temp=300,
    surface_gravity=9.5,
    stellar_radius=1.0,
)

# Atmosphere with biosignatures
oxygen_rich = {
    "N2": 0.70,
    "O2": 0.25,  # HIGH - suggests life
    "O3": 0.01,  # Ozone (product of O2)
    "H2O": 0.04,
}

simulator = AtmosphereSimulator(earth_with_oxygen)
atm = simulator.simulate_atmosphere(oxygen_rich)
wl, td, unc = simulator.generate_spectrum(atm, noise_level=1e-4)

print(f"✓ Generated atmosphere with O₂ = {oxygen_rich['O2']:.1%}")
print(f"✓ Spectrum: {len(wl)} wavelength points")
print(f"✓ Noise level: {np.mean(unc)*100:.4f}%")

print("\n2. TRAIN PINN MODEL")
print("-" * 60)
print("(Would take 8-24 hours on GPU with 10,000 samples)")
print("Commands:")
print("  $ python scripts/generate_synthetic_data.py --num_atmospheres 10000")
print("  $ python scripts/train_pinn.py --config configs/gpu.yaml")

print("\n3. DETECT BIOSIGNATURES")
print("-" * 60)

# Simulate trained model predictions
predicted_composition = torch.tensor([[0.70, 0.25, 0.01, 0.04, 0., 0., 0., 0., 0., 0.]])
predicted_composition = torch.softmax(torch.randn(1, 10), dim=1)

# Get uncertainty
pinn.eval()
with torch.no_grad():
    spectrum_tensor = torch.from_numpy(td).float().unsqueeze(0)
    mean_pred, std_pred = pinn.predict_with_uncertainty(spectrum_tensor, n_samples=10)

print(f"✓ Mean predicted composition: {mean_pred[0].numpy()[:4]}")
print(f"✓ Uncertainties (std):        {std_pred[0].numpy()[:4]}")

print("\n4. QUANTIFY LIFE PROBABILITY")
print("-" * 60)

# Biosignature detection thresholds
o2_threshold = 0.01  # 1% O2 is suspicious
o2_detected = mean_pred[0, 1].item() > o2_threshold
o2_probability = (1 - np.exp(-mean_pred[0, 1].item() / o2_threshold)) * 100

print(f"O₂ detection: {o2_detected}")
print(f"O₂ probability: {o2_probability:.1f}%")
print(f"Confidence: {'HIGH' if o2_probability > 80 else 'MEDIUM' if o2_probability > 50 else 'LOW'}")

print("\n" + "="*60)
print("✓ Workflow complete!")
print("="*60)

## Summary & Next Steps

### What You've Learned
1. ✅ What exoplanets are and why we study them
2. ✅ What biosignatures are and how they indicate life
3. ✅ How transmission spectroscopy works (the physics)
4. ✅ Why physics-informed neural networks are needed
5. ✅ The complete CosmicML-Biodetect workflow

### Next: Hands-On Training

Ready to train your own model?

```bash
# 1. Generate 10,000 synthetic atmospheres (~1 hour)
python scripts/generate_synthetic_data.py \
  --num_atmospheres 10000 \
  --output_dir data/simulated/

# 2. Train PINN model (~8-24 hours)
python scripts/train_pinn.py \
  --config configs/gpu.yaml \
  --data_dir data/simulated/

# 3. Monitor training
tensorboard --logdir models/logs/
```

### Questions to Explore
1. How much O₂ is needed to indicate life?
2. Can we distinguish biological O₂ from abiotic sources?
3. How does planet size affect biosignature detection?
4. What about planets around different star types?

### References
See docstrings in code modules for full citations.

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║     CosmicML-Biodetect: Detecting Life Among the Stars        ║
║                                                                ║
║  From synthetic data generation to biosignature detection,    ║
║  combining physics-informed AI with exoplanet atmospheres.   ║
║                                                                ║
║  Next Notebook: 02_Data_Generation.ipynb                     ║
║  Learn how to generate realistic synthetic exoplanets        ║
╚════════════════════════════════════════════════════════════════╝
""")